# Chapter 03: Validation and Generalization

[Course index](../README.md) · [Previous](../02_batches_and_epochs/README.md) · [Next](../04_optimizers_and_momentum/README.md)

**Goal:** Separate training, validation, and test roles; investigate incorrect targets and choose checkpoints.

**Study order:** read Chapter 02 first, then run this notebook from top to bottom.
This notebook contains its own setup and can run in a fresh kernel.

Before each experiment, record your prediction; afterwards, explain the output.
Related revision checkpoints: **6–8** in the
[revision notebook](../revision/01_training_basics_revision.ipynb).

## Contents

- [01. Training loss and validation loss](#lesson-01)
- [02. Track training and validation loss over epochs](#lesson-02)
- [03. Fit the supplied targets: original versus noisy data](#lesson-03)
- [04. Early stopping and patience](#lesson-04)
- [05. Evaluate selected checkpoints on a held-out test set](#lesson-05)


## Setup

Run this cell first in a fresh kernel. It imports PyTorch and recreates the original
four examples, so this chapter does not depend on another notebook's running kernel.


In [ ]:
import torch

# Original four examples; each x is paired with the y at the same position.
x = torch.tensor([0., 1., 2., 3.])
y = torch.tensor([2., 5., 8., 11.])
print("PyTorch version:", torch.__version__)
print("Inputs:", x.tolist())
print("Targets:", y.tolist())


## Prepare the model from Chapter 02

The first validation lesson evaluates the batch-size-2 model from Chapter 02.
This cell recreates that one-epoch experiment locally: two updates ending near
weight 2.795, bias 0.920. Run it before lesson 01. It uses only training examples.


In [ ]:
# Reset to compare with the other batch sizes after one epoch.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
learning_rate = 0.05

for start in range(0, len(x), 2):
    # Select two inputs and their corresponding targets.
    batch_x = x[start:start + 2]
    batch_y = y[start:start + 2]

    w.grad = None
    b.grad = None

    # Combine both examples BEFORE calculating gradients and updating.
    predictions = w * batch_x + b
    loss = ((predictions - batch_y) ** 2).mean()
    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    print(
        f"Inputs: {batch_x.tolist()} | "
        f"weight={w.item():.4f} | bias={b.item():.4f}"
    )

with torch.no_grad():
    # Use the same full-dataset metric as the batch-size-1 experiment.
    total_loss = ((w * x + b - y) ** 2).mean()

mini_batch_result = (w.item(), b.item(), total_loss.item())
print("Loss across all examples:", total_loss.item())


## Lesson 01: Training loss and validation loss

<a id="lesson-01"></a>


Training loss measures error on examples used for parameter updates. Validation loss
measures error on reserved examples that did not contribute to those updates.
It helps us check whether the learned pattern works on unseen inputs.

**Run the preparation cell immediately before this lesson.** It trains `w` and `b` for one epoch
with batch size 2, ending at approximately weight `2.795`, bias `0.920`.
The training inputs are `[0, 1, 2, 3]`; the validation inputs are `[4, 5]` with targets
`[14, 17]`. Both sets follow `y = 3 × x + 2`, but the validation inputs extend beyond
the training input range.

Use the same trained parameters for both evaluations. This cell only predicts and
measures loss: it has no `backward()` call and no parameter updates. `torch.no_grad()`
also avoids tracking the evaluation calculations for gradients.

**Predict first:** will training and validation loss be identical? Record your reason
before running. A higher validation loss in this single measurement does not establish
overfitting; we have not yet measured how both losses change across training.

### Experiment notes

- **Configuration:** one epoch, batch size 2, learning rate 0.05; forward example order.
- **Prediction before running:** _Write your prediction here._
- **Observed results:** _Record both losses and the validation predictions._
- **Explanation:** _What does the result tell you about the learned parameters?_


In [ ]:
# Reserved examples: these did not contribute to training updates.
validation_x = torch.tensor([4., 5.])
validation_y = torch.tensor([14., 17.])

with torch.no_grad():
    # Measure performance on the examples used for training.
    training_predictions = w * x + b
    training_loss = ((training_predictions - y) ** 2).mean()

    # Use the SAME parameters on unseen examples, without updating them.
    validation_predictions = w * validation_x + b
    validation_loss = (
        (validation_predictions - validation_y) ** 2
    ).mean()

print("Weight:", w.item())
print("Bias:", b.item())
print("Validation predictions:", validation_predictions.tolist())
print("Training loss:", training_loss.item())
print("Validation loss:", validation_loss.item())


## Lesson 02: Track training and validation loss over epochs

<a id="lesson-02"></a>


**Prediction before running:** training loss will decrease while validation loss increases.

Test this prediction with the same four training examples, two reserved validation
examples, batch size 2, learning rate 0.05, and forward example order. The changed
setting is training duration: we train for 100 epochs and evaluate after each epoch.
Run the setup cell first to create `x` and `y`; this cell resets its own parameters and
validation examples. Its first epoch should reproduce sections 14 and 17.

Reset parameters only once, before the outer epoch loop. Each epoch has two batches
and therefore two updates. Across 100 epochs, this experiment makes 200 updates.
The validation examples never contribute to gradients or parameter updates.

A higher validation loss at one point does not by itself establish overfitting.
Inspect how both losses change across epochs. Training can improve performance on
unseen examples; overfitting is a possible outcome, not an automatic consequence of
training for more epochs. These validation inputs follow the same noiseless linear
rule as the training inputs.

### Experiment notes

- **Prediction:** training loss falls while validation loss rises.
- **Observation:** _Record how both losses change between epochs 1, 10, and 100._
- **Explanation:** _Did the result support the prediction? How close are the final parameters to weight 3 and bias 2?_


In [ ]:
# Reset once before training, not at the beginning of every epoch.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
learning_rate = 0.05
batch_size = 2

# Reserve these examples for evaluation only.
validation_x = torch.tensor([4., 5.])
validation_y = torch.tensor([14., 17.])
history = []

for epoch in range(1, 101):
    # One epoch visits all four training examples in two batches.
    for start in range(0, len(x), batch_size):
        batch_x = x[start:start + batch_size]
        batch_y = y[start:start + batch_size]

        w.grad = None
        b.grad = None
        predictions = w * batch_x + b
        loss = ((predictions - batch_y) ** 2).mean()
        loss.backward()

        with torch.no_grad():
            w -= learning_rate * w.grad
            b -= learning_rate * b.grad

    # Evaluate both datasets at the SAME parameter values after this epoch.
    # This block makes predictions and measures loss without updating anything.
    with torch.no_grad():
        training_loss = ((w * x + b - y) ** 2).mean().item()
        validation_loss = (
            (w * validation_x + b - validation_y) ** 2
        ).mean().item()

    # Save every epoch so we can inspect the complete learning history later.
    history.append((epoch, training_loss, validation_loss))

    # Print checkpoints to keep the notebook output short and readable.
    if epoch in (1, 2, 5, 10, 20, 50, 100):
        print(
            f"Epoch {epoch:3d} | training loss={training_loss:.6f} | "
            f"validation loss={validation_loss:.6f}"
        )

print(f"Final weight: {w.item():.6f} | Final bias: {b.item():.6f}")


## Lesson 03: Fit the supplied targets: original versus noisy data

<a id="lesson-03"></a>


**Question:** can fitting the supplied training answers more closely make performance
on unseen examples worse? The loss uses the supplied target; it cannot know whether
we supplied a wrong answer.

Make an independent copy of `y` using `clone()`, then change the answer for input 3
from 11 to 14. The original `y` stays unchanged. The reserved validation examples
continue to follow the original rule `y = 3 × x + 2`.

Compare two runs with identical settings: starting weight 1, bias 0, learning rate
0.05, batch size 2, forward order, and 100 epochs. **Only the training targets differ.**
Run the setup cell first to import PyTorch and create `x` and `y`.

Each run's training loss is measured against its own training targets. Both validation
losses use the same correct validation targets. Look at the trend across epochs, rather
than assuming every metric must decrease at every epoch: mini-batch updates follow
individual batch gradients, so full-dataset training loss can fluctuate too.

Whenever validation loss reaches a new low, save a checkpoint: the epoch, validation
loss, weight, and bias. Saving `.item()` values captures the numbers at that moment.
We continue through all 100 epochs so we can see how later training compares with the
best saved checkpoint; this experiment does not stop training early.

### Experiment notes

- **Prediction:** the original targets should support the true pattern; the changed target may pull the model away from it as training continues.
- **Observation:** _Compare each run's validation loss at epochs 1, 3, and 100. Which checkpoint had the lowest validation loss?_
- **Explanation:** _Why can a model improve its fit to incorrect training answers while making worse predictions on correct unseen examples?_


In [ ]:
# Preserve the original targets; modify only an independent copy.
noisy_y = y.clone()
noisy_y[3] = 14.

# These correct targets never contribute to training updates.
validation_x = torch.tensor([4., 5.])
validation_y = torch.tensor([14., 17.])
learning_rate = 0.05
batch_size = 2
experiment_results = {}

for label, training_targets in [
    ("Original targets", y),
    ("Changed target", noisy_y),
]:
    # Reset each run so both start from exactly the same parameters.
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)
    run_history = []
    best_checkpoint = None
    print(f"\n{label}: {training_targets.tolist()}")

    for epoch in range(1, 101):
        for start in range(0, len(x), batch_size):
            batch_x = x[start:start + batch_size]
            batch_y = training_targets[start:start + batch_size]

            w.grad = None
            b.grad = None
            loss = ((w * batch_x + b - batch_y) ** 2).mean()
            loss.backward()

            with torch.no_grad():
                w -= learning_rate * w.grad
                b -= learning_rate * b.grad

        # Evaluate both losses after the epoch without changing the parameters.
        with torch.no_grad():
            training_loss = ((w * x + b - training_targets) ** 2).mean().item()
            validation_loss = (
                (w * validation_x + b - validation_y) ** 2
            ).mean().item()

        run_history.append((epoch, training_loss, validation_loss))

        # Choose checkpoints by VALIDATION loss, not training loss.
        if best_checkpoint is None or validation_loss < best_checkpoint["validation_loss"]:
            best_checkpoint = {
                "epoch": epoch,
                "validation_loss": validation_loss,
                "weight": w.item(),
                "bias": b.item(),
            }

        if epoch in (1, 2, 3, 5, 10, 20, 50, 100):
            print(
                f"Epoch {epoch:3d} | training loss={training_loss:.6f} | "
                f"validation loss={validation_loss:.6f}"
            )

    # Retain each run's learning history and best checkpoint for later reference.
    experiment_results[label] = {
        "history": run_history,
        "best_checkpoint": best_checkpoint,
    }
    print(f"Final weight: {w.item():.6f} | Final bias: {b.item():.6f}")
    print("Best checkpoint by validation loss:", best_checkpoint)


## Lesson 04: Early stopping and patience

<a id="lesson-04"></a>


**Prediction:** patience 2 usually stops sooner than patience 50 because it tolerates
fewer consecutive epochs without a new best validation loss.

Run the setup cell first to import PyTorch and create the original `x` and `y`.
Compare patience 2 and 50 on both original and changed targets. Every run starts at
weight 1, bias 0, with batch size 2, learning rate 0.05, forward order, and at most
100 epochs. Only the patience setting differs within each dataset comparison.
Patience 50 is an illustrative choice for this experiment, not a universal recommendation.

The early-stopping counter tracks **epochs without a new best validation loss**, not
just epochs where loss rises from the immediately preceding epoch. Any new best resets
the counter to zero. When the counter reaches patience, stop and restore the best
checkpoint. The parameters from the stopping epoch may differ from the best checkpoint.

Stopping early saves updates, but short patience can miss a later recovery. Recall
that the original-target run had a very low validation loss at epoch 2, followed by a
temporary rise, yet eventually performed better at epoch 100.

### Experiment notes

- **Prediction:** _Which dataset might benefit from waiting longer?_
- **Observation:** _For each run, record its stopping epoch, best epoch, and restored validation loss._
- **Explanation:** _Why might short patience help with the changed target but stop the original-target run too soon?_


In [ ]:
def train_with_early_stopping(training_targets, patience, max_epochs=100):
    """Train until patience is exhausted or the epoch limit is reached."""
    # Local parameters give every experiment an independent, identical start.
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)
    learning_rate = 0.05
    batch_size = 2
    validation_x = torch.tensor([4., 5.])
    validation_y = torch.tensor([14., 17.])

    best_checkpoint = None
    epochs_without_improvement = 0
    history = []
    stop_reason = "Reached epoch limit"

    for epoch in range(1, max_epochs + 1):
        for start in range(0, len(x), batch_size):
            batch_x = x[start:start + batch_size]
            batch_y = training_targets[start:start + batch_size]

            w.grad = None
            b.grad = None
            loss = ((w * batch_x + b - batch_y) ** 2).mean()
            loss.backward()

            with torch.no_grad():
                w -= learning_rate * w.grad
                b -= learning_rate * b.grad

        # Validation only measures performance; it never updates parameters.
        with torch.no_grad():
            training_loss = ((w * x + b - training_targets) ** 2).mean().item()
            validation_loss = (
                (w * validation_x + b - validation_y) ** 2
            ).mean().item()
        history.append((epoch, training_loss, validation_loss))

        if best_checkpoint is None or validation_loss < best_checkpoint["validation_loss"]:
            # Save independent numeric values, not references to changing tensors.
            best_checkpoint = {
                "epoch": epoch,
                "validation_loss": validation_loss,
                "weight": w.item(),
                "bias": b.item(),
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            stop_reason = "Reached patience"
            break

    # Restore the BEST checkpoint, not necessarily the last trained parameters.
    with torch.no_grad():
        w.fill_(best_checkpoint["weight"])
        b.fill_(best_checkpoint["bias"])
        restored_validation_loss = (
            (w * validation_x + b - validation_y) ** 2
        ).mean().item()

    return {
        "stopped_epoch": epoch,
        "stop_reason": stop_reason,
        "best_checkpoint": best_checkpoint,
        "restored_validation_loss": restored_validation_loss,
        "history": history,
    }


# Keep the original data intact and create the same changed target as lesson 3.
noisy_y = y.clone()
noisy_y[3] = 14.
early_stopping_results = {}

for label, targets in [("Original targets", y), ("Changed target", noisy_y)]:
    for patience in (2, 50):
        result = train_with_early_stopping(targets, patience)
        early_stopping_results[(label, patience)] = result
        best = result["best_checkpoint"]
        print(
            f"{label} | patience={patience:2d} | "
            f"stopped epoch={result['stopped_epoch']:3d} | "
            f"best epoch={best['epoch']:3d} | "
            f"restored validation loss={result['restored_validation_loss']:.6f}"
        )
        print("Stop reason:", result["stop_reason"])


## Lesson 05: Evaluate selected checkpoints on a held-out test set

<a id="lesson-05"></a>


| Dataset | Purpose |
|---|---|
| Training | Calculate gradients and update parameters |
| Validation | Choose settings and checkpoints |
| Test | Evaluate the chosen model after those decisions |

Validation examples do not produce parameter updates, but they influence our decisions.
A separate test set provides a final check on examples reserved from both training and
validation-based selection. Do not repeatedly tune settings to improve the test result;
that would make the test set part of the selection process too.

We already selected the original-target checkpoint with patience 50, and the changed-target
checkpoint with patience 2, using lesson 4's validation results. These choices are fixed
before this evaluation. Run the setup cell, then lesson 4, before this cell.

Reserve test inputs `[6, 7]` with correct answers `[20, 23]`, following the same rule
`y = 3 × x + 2`. Their inputs extend beyond the training and validation ranges. Evaluate
both selected checkpoints without gradients or parameter updates. The test loss describes
performance on these two examples; a small test set does not establish performance on all
possible inputs.

### Experiment notes

- **Prediction before running:** _How will each selected checkpoint perform on these test inputs?_
- **Observation:** _Record both predictions and each test loss._
- **Explanation:** _Why might a checkpoint with small validation loss still have a larger loss on test inputs farther from the training range?_


In [ ]:
# New examples: reserved from training updates and checkpoint selection.
test_x = torch.tensor([6., 7.])
test_y = torch.tensor([20., 23.])
test_results = {}

# These configurations were chosen using validation loss, before checking test loss.
selected_runs = [("Original targets", 50), ("Changed target", 2)]

for label, patience in selected_runs:
    checkpoint = early_stopping_results[(label, patience)]["best_checkpoint"]

    # Load the saved numeric values, rather than using the last global w and b.
    test_w = torch.tensor(checkpoint["weight"])
    test_b = torch.tensor(checkpoint["bias"])

    with torch.no_grad():
        test_predictions = test_w * test_x + test_b
        test_loss = ((test_predictions - test_y) ** 2).mean().item()

    test_results[label] = {
        "selected_epoch": checkpoint["epoch"],
        "predictions": test_predictions.tolist(),
        "test_loss": test_loss,
    }
    print(f"\n{label} | selected epoch={checkpoint['epoch']}")
    print("Test inputs:", test_x.tolist())
    print("Correct answers:", test_y.tolist())
    print("Test predictions:", test_predictions.tolist())
    print(f"Test loss: {test_loss:.8f}")
